# Teks — NER / pelabelan token dengan transformer (BIO, perataan subword, F1 entitas)
**Kapan dipakai:** label per KATA, bukan per teks: entitas bernama (orang, tempat, organisasi),
POS tag, ekstraksi atribut (merek, ukuran, harga dari judul produk), slot pengisian. `ner`
(CRF) = jalur cepat tanpa GPU; ini jalur skor tinggi.

**Isi:** data format panjang (satu baris = satu kata: `sentence_id`, `token`, `tag`) → kalimat →
tokenizer `is_split_into_words` → label hanya di subword pertama tiap kata (sisanya -100,
diabaikan loss) → `AutoModelForTokenClassification` → prediksi per kata dari subword pertama →
F1 entitas (span harus persis sama, seperti seqeval) ditulis sendiri.

**Alur:** Konfigurasi → baca → kalimat → split validasi per kalimat → latih (pilih epoch dari F1
entitas) → laporan per jenis entitas → prediksi test → submission per kata → validator.

**Saat lomba yang diubah:** sel Konfigurasi. Format soal lain (satu baris = satu kalimat dengan
kolom tag dipisah spasi): ubah sel Baca saja, sisanya sama.

**Data demo:** NERP (`ner`, IndoNLU, 5 jenis entitas: PPL, PLC, IND, EVT, FNB).

In [ ]:
import os
import time

import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from torch.nn.functional import cross_entropy
from torch.nn.utils import clip_grad_norm_
from transformers import AutoModelForTokenClassification, AutoTokenizer
from transformers import get_linear_schedule_with_warmup

## Konfigurasi

In [ ]:
DATA_FOLDER = "../../../data_latihan/text/ner"                # GANTI saat lomba
TRAIN_CSV = f"{DATA_FOLDER}/nerp_train.csv"                    # GANTI saat lomba
TEST_CSV = f"{DATA_FOLDER}/nerp_test.csv"                      # GANTI saat lomba
OUTPUT_FOLDER = "../../../outputs/text/ner_bert"
SUBMISSION_NAME = "submission.csv"
SENT_COL = "sentence_id"         # GANTI saat lomba: nomor kalimat
WORD_COL = "token"               # GANTI saat lomba: kata
TAG_COL = "tag"                  # GANTI saat lomba: label BIO (B-XXX, I-XXX, O)
OUTSIDE = "O"
VAL_SHARE = 0.1                  # bagian kalimat train untuk validasi (pilih epoch)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_BF16 = DEVICE == "cuda"

In [ ]:
PRETRAINED = "../../../pretrained/indobert"           # GANTI saat lomba
# PRETRAINED = "../../../pretrained/xlmr_base"        # campur bahasa / huruf besar-kecil penting
# PRETRAINED = "../../../pretrained/indobert_large"   # terkuat: BATCH_SIZE 8
MAX_LENGTH = 256                 # subword per kalimat, kata sesudahnya diberi OUTSIDE
BATCH_SIZE = 16
EPOCHS = 5
LEARNING_RATE = 5e-5             # NER biasanya suka lr sedikit lebih besar dari klasifikasi
WARMUP = 0.1
SEED = 42

## Baca dan susun kalimat
Urutan kata di dalam kalimat mengikuti urutan baris file. `test_order` menyimpan posisi baris
asli agar submission urut sama dengan file test.

In [ ]:
os.makedirs(OUTPUT_FOLDER, exist_ok=True)
train = pd.read_csv(TRAIN_CSV, keep_default_na=False)     # kata "NA"/"null" tetap teks
test = pd.read_csv(TEST_CSV, keep_default_na=False)


def to_sentences(frame, with_tags):
    words, tags, positions = [], [], []
    for _, part in frame.groupby(SENT_COL, sort=False):
        words.append(part[WORD_COL].astype(str).tolist())
        positions.append(part.index.to_numpy())
        if with_tags:
            tags.append(part[TAG_COL].astype(str).tolist())
    return words, tags, positions


train_words, train_tags, _ = to_sentences(train, True)
test_words, _, test_positions = to_sentences(test, False)
tag_names = sorted(train[TAG_COL].astype(str).unique())
tag_to_id = {}
for number, name in enumerate(tag_names):
    tag_to_id[name] = number
print("kalimat train:", len(train_words), "| test:", len(test_words), "| tag:", tag_names)

## F1 entitas
Entitas = span (jenis, awal, akhir). Benar hanya bila jenis DAN batasnya persis sama. `I-` tanpa
`B-` di depannya dianggap awal entitas baru (aturan seqeval).

In [ ]:
def spans(tags):
    found, start, kind = set(), None, None
    for position, tag in enumerate(tags + [OUTSIDE]):
        continues = tag.startswith("I-") and tag[2:] == kind
        if start is not None and not continues:
            found.add((kind, start, position))
            start = None
        if tag.startswith("B-") or (tag.startswith("I-") and start is None):
            start, kind = position, tag[2:]
    return found


print(spans(["B-PPL", "I-PPL", "O", "I-PLC", "B-PLC"]))   # 1 PPL (0-2), 2 PLC (3-4, 4-5)

In [ ]:
def keep_kind(found, only_kind):
    if only_kind is None:
        return found
    kept = set()
    for item in found:
        if item[0] == only_kind:
            kept.add(item)
    return kept


def entity_counts(truth_list, pred_list, only_kind=None):
    hit, wanted, given = 0, 0, 0
    for truth, pred in zip(truth_list, pred_list):
        true_set = keep_kind(spans(truth), only_kind)
        pred_set = keep_kind(spans(pred), only_kind)
        hit += len(true_set & pred_set)
        wanted += len(true_set)
        given += len(pred_set)
    return hit, wanted, given


def entity_f1(truth_list, pred_list, only_kind=None):
    hit, wanted, given = entity_counts(truth_list, pred_list, only_kind)
    return 2 * hit / max(1, wanted + given)


print("entity_f1 siap")

## Tokenisasi dan perataan label
`firsts` = pasangan (nomor kata, posisi subword pertamanya). Hanya posisi itu yang diberi label;
subword lanjutan, [CLS], [SEP], dan padding diberi -100 (diabaikan loss).

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(PRETRAINED, local_files_only=True)


def align(word_ids, tags):
    labels, firsts, previous = [], [], None
    for position, word in enumerate(word_ids):
        label = -100
        if word is not None and word != previous:
            firsts.append((word, position))
            if tags is not None:
                label = tag_to_id.get(tags[word], 0)
        labels.append(label)
        previous = word
    return labels, firsts


print("align siap")

In [ ]:
def encode_sentences(words_list, tags_list):
    encoded = tokenizer(words_list, is_split_into_words=True, truncation=True,
                        max_length=MAX_LENGTH)
    rows, labels_list, firsts_list = [], [], []
    for number in range(len(words_list)):
        tags = None
        if tags_list is not None:
            tags = tags_list[number]
        labels, firsts = align(encoded.word_ids(number), tags)
        rows.append({"input_ids": encoded["input_ids"][number],
                     "attention_mask": encoded["attention_mask"][number]})
        labels_list.append(labels)
        firsts_list.append(firsts)
    return rows, labels_list, firsts_list


print("encode_sentences siap")

In [ ]:
train_rows, train_labels, train_firsts = encode_sentences(train_words, train_tags)
test_rows, _, test_firsts = encode_sentences(test_words, None)
cut_words = 0
for words, firsts in zip(train_words, train_firsts):
    cut_words += len(words) - len(firsts)
print("kata tanpa subword (terpotong MAX_LENGTH / kosong):", cut_words)
fit_ids, val_ids = train_test_split(np.arange(len(train_rows)), test_size=VAL_SHARE,
                                    random_state=SEED)
print("kalimat latih:", len(fit_ids), "| validasi:", len(val_ids))

## Batch, latih, prediksi per kata

In [ ]:
def pick(items, chosen):
    picked = []
    for number in chosen:
        picked.append(items[number])
    return picked


def make_batch(chosen, rows, labels_list):
    batch = tokenizer.pad(pick(rows, chosen), return_tensors="pt").to(DEVICE)
    labels = np.full((len(chosen), batch["input_ids"].shape[1]), -100, dtype="int64")
    for row_number, number in enumerate(chosen):
        sequence = labels_list[number]
        labels[row_number, :len(sequence)] = sequence
    return batch, torch.tensor(labels).to(DEVICE)


print("make_batch siap")

In [ ]:
def train_epoch(model, optimizer, scheduler, rng):
    model.train()
    order = rng.permutation(fit_ids)
    for start in range(0, len(order), BATCH_SIZE):
        batch, labels = make_batch(order[start:start + BATCH_SIZE], train_rows, train_labels)
        with torch.autocast("cuda", dtype=torch.bfloat16, enabled=USE_BF16):
            logits = model(**batch).logits.float()
        loss = cross_entropy(logits.reshape(-1, len(tag_names)), labels.reshape(-1),
                             ignore_index=-100)
        loss.backward()
        clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()


print("train_epoch siap")

In [ ]:
def batch_tags(best, chosen, firsts_list, words_list):
    result = []
    for row_number, number in enumerate(chosen):
        tags = [OUTSIDE] * len(words_list[number])
        for word, position in firsts_list[number]:
            tags[word] = tag_names[best[row_number, position]]
        result.append(tags)
    return result


def predict_tags(model, chosen_ids, rows, firsts_list, words_list):
    model.eval()
    result = []
    with torch.no_grad():
        for start in range(0, len(chosen_ids), BATCH_SIZE * 4):
            chosen = chosen_ids[start:start + BATCH_SIZE * 4]
            batch = tokenizer.pad(pick(rows, chosen), return_tensors="pt").to(DEVICE)
            with torch.autocast("cuda", dtype=torch.bfloat16, enabled=USE_BF16):
                logits = model(**batch).logits
            best = logits.argmax(dim=-1)
            best = best.cpu().numpy()
            result = result + batch_tags(best, chosen, firsts_list, words_list)
    return result


print("predict_tags siap")

## Latih
Bobot epoch dengan F1 entitas validasi tertinggi disimpan.

In [ ]:
torch.manual_seed(SEED)
model = AutoModelForTokenClassification.from_pretrained(
    PRETRAINED, num_labels=len(tag_names), ignore_mismatched_sizes=True, local_files_only=True)
model = model.to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
steps = int(np.ceil(len(fit_ids) / BATCH_SIZE)) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, int(WARMUP * steps), steps)
rng = np.random.default_rng(SEED)
val_true = pick(train_tags, val_ids)
best_f1, best_state = -1.0, None
began = time.time()
for epoch in range(EPOCHS):
    train_epoch(model, optimizer, scheduler, rng)
    val_pred = predict_tags(model, val_ids, train_rows, train_firsts, train_words)
    value = entity_f1(val_true, val_pred)
    if value > best_f1:
        best_f1, best_state = value, {}
        for key, tensor in model.state_dict().items():
            best_state[key] = tensor.detach().clone()
    print(f"epoch {epoch + 1}: F1 entitas validasi {value:.4f} | {time.time() - began:.0f} dtk")
model.load_state_dict(best_state)

## Laporan per jenis entitas (validasi)

In [ ]:
val_pred = predict_tags(model, val_ids, train_rows, train_firsts, train_words)
kinds = set()
for tags in val_true:
    for kind, _, _ in spans(tags):
        kinds.add(kind)
kinds = sorted(kinds)
f1_values, support = [], []
for kind in kinds:
    f1_values.append(entity_f1(val_true, val_pred, kind))
    support.append(entity_counts(val_true, val_pred, kind)[1])
report = pd.DataFrame()
report["entitas"] = kinds + ["SEMUA"]
report["f1"] = f1_values + [entity_f1(val_true, val_pred)]
report["jumlah_benar"] = support + [sum(support)]
print(report.round(4).to_string(index=False))

## Prediksi test dan submission
Satu baris per kata, urutan sama dengan file test. Soal minta satu baris per kalimat: gabungkan
tag tiap kalimat di `test_pred` dengan spasi.

In [ ]:
test_pred = predict_tags(model, np.arange(len(test_rows)), test_rows, test_firsts, test_words)
predicted = np.empty(len(test), dtype=object)
for positions, tags in zip(test_positions, test_pred):
    predicted[positions] = tags

submission = pd.DataFrame()
submission[SENT_COL] = test[SENT_COL]
submission[WORD_COL] = test[WORD_COL]
submission[TAG_COL] = predicted
submission.to_csv(f"{OUTPUT_FOLDER}/{SUBMISSION_NAME}", index=False)
tag_share = submission[TAG_COL].value_counts()
print(tag_share.head(10).to_dict())
submission.head()

## Cek submission
Jumlah baris = baris test, tidak ada tag kosong, tag hanya dari daftar tag train.

In [ ]:
written = pd.read_csv(f"{OUTPUT_FOLDER}/{SUBMISSION_NAME}", keep_default_na=False)
problems = []
if len(written) != len(test):
    problems.append(f"baris {len(written)}, test {len(test)}")
if (written[TAG_COL] == "").any():
    problems.append("ada tag kosong")
foreign = set(written[TAG_COL]) - set(tag_names)
if foreign:
    problems.append(f"tag asing {sorted(foreign)[:5]}")
for problem in problems:
    print("-", problem)
print("MASALAH:", len(problems))

## Cek dengan kunci jawaban (HAPUS saat lomba)
File test demo NERP masih memuat kolom tag asli.

In [ ]:
_, test_true, _ = to_sentences(test, True)
print(f"F1 entitas test (kunci): {entity_f1(test_true, test_pred):.4f} | validasi {best_f1:.4f}")